In [ ]:
from typing import Tuple
from collections import deque
import textstat
import os
from tree.tree import Tree
from tree.node import Node, TerminalNode
from adapters.SemanticPerturb import PromptPackage
from adapters.TerminalPerturb import TerminalPerturber
from similarity.cosine_similarity import similarity
from model.engine import LLMAdapter
from adapters.OAI_Embeddings import RobertaEmbedder
import utils.constants as constants
from itertools import repeat
import utils.constants as constants
from tree.tree import ReadTree

2.6.0+cu124


In [144]:
gen_modelIds = constants.MODELS

# track pre-check trees


In [ ]:
POPQA_valid_ids = {
    'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;question_position_middle, question_position_suffix;sg_dialect, sg_dialect, sg_dialect;question_position_middle, sg_dialect;question_position_suffix, sg_dialect;question_position_suffix;question_position_middle',
    'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;question_position_middle, question_position_suffix;sg_dialect, sg_dialect',
    'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;sg_dialect, sg_dialect',
}


Term 'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;question_position_middle, question_position_suffix;sg_dialect, sg_dialect, sg_dialect;question_position_middle, sg_dialect;question_position_suffix, sg_dialect;question_position_suffix;question_position_middle'
Term 'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;question_position_middle, question_position_suffix;sg_dialect, sg_dialect'
Term 'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;sg_dialect, sg_dialect'


# check TQA progress

In [120]:
targets = {'question_position_middle;sg_dialect', 'question_position_suffix;sg_dialect'}

# check = []
# inter_dir = f'{constants.TREE_DIR}{dataset_name}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
# for filename in os.listdir(inter_dir):
#     if filename.endswith('.pkl'):
#         check.append(int(filename[:-len('.pkl')]))
# print(f"Number of trees to check: {len(check)}")

for model in gen_modelIds:
    print(f"\nModel: {model}")
    has_ans_count = 0
    no_ans_count = 0

    for tree_id in check_gen_tree_ids:
        tree_path = f"{constants.TREE_DIR}{dataset_name}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/{model.replace('/', '-')}/complete/{tree_id}_checked.pkl"
        try:
            tree = ReadTree.load_read_tree(tree_path)
            for child in tree.root.children:
                total_children = len(tree.root.children)
                print(total_children)
                if isinstance(child, TerminalNode) and child.metadata.get("terminal_name") in targets:
                    if child.answers:
                        has_ans_count += 1
                    else:
                        no_ans_count += 1
        except Exception:
            continue

    print(f"  With answers: {has_ans_count}")
    print(f"  Without answers: {no_ans_count}")



Model: google/gemma-3-1b-it
  With answers: 0
  Without answers: 0

Model: google/gemma-3-12b-it
  With answers: 0
  Without answers: 0

Model: mistralai/Mistral-7B-Instruct-v0.2
  With answers: 0
  Without answers: 0


In [190]:
'''
each question, answer tree has new value:
- is terminal (node)
- terminal_preturb applied (list of string)
'''
from collections import defaultdict
# ########### set ###########
terminal_type = 'sg_dialect'
dataset_name = "POPQA"
strategy_path = "para-prefix"
# ###########################
missing_tree_path = []
gen_modelIds = constants.MODELS
VALID_TERMS = {'question_position_suffix;sg_dialect', 'question_position_middle;sg_dialect'}

def stringify_terminal_names(root):
    term_names = set()
    children = root.children
    for root in children:
        term_name =  root.metadata.get("terminal_name", '')
        if term_name:        
            term_names.add(str(term_name))
    term_names = list(term_names)
    term_names.sort()
    return ', '.join(str(item) for item in term_names)

# maps term -> set of tree_ids
pop_applied_map = defaultdict(set)

tree_ids = []
sus_tree = []
inter_dir = f'{constants.TREE_DIR}{dataset_name}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
for filename in os.listdir(inter_dir):
    if filename.endswith('.pkl'):
        tree_ids.append(int(filename[: -len('.pkl')]))
tree_ids.sort()
print(tree_ids)
# load and process all trees
for tree_id in tree_ids:
    try:
        tree = ReadTree.load_read_tree(f"{inter_dir}{tree_id}.pkl")
        term = stringify_terminal_names(tree.root)
        pop_applied_map[term].add(tree_id)
    except Exception as e:
        print(f"Error processing tree {tree_id}: {e}")
        sus_tree.append(tree_id)
        continue
for term, ids in pop_applied_map.items():
    print(f"Term '{term}', trees = ({len(ids)}) appears in trees: {sorted(ids)}")

[23, 25, 70, 103, 106, 228, 287, 375, 402, 447, 470, 486, 503, 601, 662, 671, 737, 739, 743, 770, 772, 827, 830, 848, 933, 935, 974, 984, 989, 1041, 1055, 1070, 1080, 1081, 1089, 1097, 1117, 1153, 1158, 1174, 1193, 1220, 1228, 1246, 1250, 1280, 1308, 1347, 1391, 1465, 1488, 1490, 1495, 1553, 1568, 1571, 1605, 1644, 1679, 1694, 1716, 1737, 1745, 1768, 1796, 1799, 1825, 1845, 1850, 1858, 1955, 1966, 2002, 2020, 2057, 2089, 2169, 2187, 2198, 2204, 2252, 2259, 2274, 2280, 2329, 2379, 2438, 2481, 2503, 2508, 2519, 2523, 2590, 2595, 2614, 2661, 2667, 2693, 2701, 2717, 2753, 2763, 2785, 2788, 2817, 2828, 2839, 2883, 2897, 2964, 2965, 3036, 3043, 3082, 3094, 3174, 3177, 3208, 3241, 3289, 3305, 3329, 3361, 3430, 3431, 3432, 3441, 3469, 3494, 3500, 3507, 3511, 3585, 3600, 3611, 3629, 3636, 3731, 3854, 3859, 3861, 3864, 3879, 3893, 3910, 3943, 3947, 3965, 3981, 4014, 4062, 4112, 4113, 4133, 4145, 4161, 4198, 4220, 4221, 4282, 4331, 4350, 4358, 4368, 4384, 4415, 4475, 4524, 4528, 4564, 4567, 4583,

In [194]:
# assume pop_applied_map is already populated
union_pop_ids = set()

# take the first three terms in insertion order (or sort if you prefer)
first_three_terms = list(pop_applied_map)[:3]

for idx, term in enumerate(first_three_terms, start=0):
    ids = pop_applied_map[term]
    print(len(ids))
    union_pop_ids |= ids
    print(f"{idx}. Term '{term}': {len(ids)} trees; cumulative union size: {len(union_pop_ids)}")

# if you want, also print the final union set itself
print(sorted(union_pop_ids))
union_pop_ids = list(union_pop_ids)

98
0. Term 'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;question_position_middle, question_position_suffix;sg_dialect, sg_dialect, sg_dialect;question_position_middle, sg_dialect;question_position_suffix, sg_dialect;question_position_suffix;question_position_middle': 98 trees; cumulative union size: 98
105
1. Term 'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;question_position_middle, question_position_suffix;sg_dialect, sg_dialect': 105 trees; cumulative union size: 203
71
2. Term 'question_position_middle, question_position_middle;sg_dialect, question_position_suffix, question_position_suffix;sg_dialect, sg_dialect': 71 trees; cumulative union size: 274
[23, 25, 70, 103, 106, 228, 375, 402, 447, 486, 503, 601, 662, 671, 737, 743, 770, 772, 827, 830, 848, 933, 974, 984, 989, 1041, 1055, 1070, 1080, 1081, 1089, 1097, 1117, 1153, 1158, 1174, 1193, 12

In [197]:
'''
each question, answer tree has new value:
- is terminal (node)
- terminal_preturb applied (list of string)
'''
from collections import defaultdict
# ########### set ###########
terminal_type = 'sg_dialect'
dataset_name = "TQA"
strategy_path = "para-prefix"
# ###########################
missing_tree_path = []
gen_modelIds = constants.MODELS
VALID_TERMS = {'question_position_suffix;sg_dialect', 'question_position_middle;sg_dialect'}

def stringify_terminal_names(root):
    term_names = set()
    children = root.children
    for root in children:
        term_name =  root.metadata.get("terminal_name", '')
        if term_name:        
            term_names.add(str(term_name))
    term_names = list(term_names)
    term_names.sort()
    return ', '.join(str(item) for item in term_names)

# maps term -> set of tree_ids
tqa_applied_map = defaultdict(set)

tree_ids = []
sus_tree = []
inter_dir = f'{constants.TREE_DIR}{dataset_name}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
for filename in os.listdir(inter_dir):
    if filename.endswith('.pkl'):
        tree_ids.append(int(filename[: -len('.pkl')]))
tree_ids.sort()
print(tree_ids)
# load and process all trees
for tree_id in tree_ids:
    try:
        tree = ReadTree.load_read_tree(f"{inter_dir}{tree_id}.pkl")
        term = stringify_terminal_names(tree.root)
        tqa_applied_map[term].add(tree_id)
    except Exception as e:
        print(f"Error processing tree {tree_id}: {e}")
        sus_tree.append(tree_id)
        continue
for term, ids in tqa_applied_map.items():
    print(f"Term '{term}', trees = ({len(ids)}) appears in trees: {sorted(ids)}")

[75, 96, 213, 225, 293, 485, 545, 547, 582, 597, 707, 722, 817, 845, 1362, 1638, 1784, 2103, 2203, 2232, 2352, 2435, 2555, 2579, 2862, 2891, 3025, 3198, 3212, 3283, 3291, 3313, 3369, 3474, 3695, 3749, 3775, 3794, 4042, 4135, 4150, 4223, 4488, 4589, 4638, 4768, 4987, 5010, 5064, 5102, 5105, 5120, 5374, 5413, 5754, 5843, 6064, 6440, 6663, 6665, 6946, 7473, 7503, 7591, 7675, 7749, 7989, 8041, 8081, 8095, 8140, 8509, 8629, 8755, 8838, 8868, 8955, 9146, 9363, 9505, 9648, 9664, 9706, 9881, 9883, 10081, 10192, 10326, 10499, 10588, 10768, 10892, 11004, 11026, 11371, 11453, 11869, 12236, 12300, 12489, 12531, 12533, 12711, 12741, 12777, 13321, 13387, 13490, 13668, 13721, 13755, 13796, 13911, 13926, 14020, 14054, 14167, 14179, 14197, 14682, 14743, 15008, 15317, 15538, 15673, 15722, 15749, 15967, 15986, 16029, 16357, 16533, 16676, 16823, 16979, 17103, 17106, 17214, 17485, 17562, 17621, 17624, 18155, 18169, 18393, 18801, 18862, 18879, 18912, 19126, 19349, 19367, 19637, 19689, 20186, 20202, 20464, 2

In [179]:
import os
from collections import defaultdict

def track(dataset_name, tree_ids=None):
    if not tree_ids:
        inter_dir = f'{constants.TREE_DIR}{dataset_name}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
        tree_ids = sorted(
            int(fn[:-4]) for fn in os.listdir(inter_dir) if fn.endswith('.pkl')
        )

    VALID_TERMS = {
        'sg_dialect',
        'question_position_suffix;sg_dialect',
        'question_position_middle;sg_dialect'
    }

    # Data structure to hold results per model
    results = {}

    for model in gen_modelIds:
        has_ans_trees = defaultdict(list)
        no_ans_trees  = defaultdict(list)

        for tree_id in tree_ids:
            tree_path = (
                f"{constants.TREE_DIR}"
                f"{dataset_name}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/"
                f"{model.replace('/', '-')}/complete/{tree_id}_checked.pkl"
            )
            try:
                tree = ReadTree.load_read_tree(tree_path)
            except Exception:
                # skip missing or bad files
                continue

            for child in tree.root.children:
                if not isinstance(child, TerminalNode):
                    continue

                term = child.metadata.get("terminal_name")
                cat  = term if term in VALID_TERMS else "others"

                # collect the tree_id in the appropriate list
                if child.answers:
                    has_ans_trees[cat].append(tree_id)
                else:
                    no_ans_trees[cat].append(tree_id)

        # Ensure every category appears, even if empty
        all_cats = list(VALID_TERMS) + ["others"]
        results[model] = {
            'has_answers': {cat: has_ans_trees.get(cat, []) for cat in all_cats},
            'no_answers' : {cat: no_ans_trees.get(cat, [])  for cat in all_cats},
        }

    return results


In [ ]:
pqa_res = track('POPQA')

In [ ]:
union_sg_popqa = set()
for model, data in pqa_res.items():
    union_sg_popqa |= set(data['has_answers']['sg_dialect'])

# get a sorted list (if you want it ordered)
union_sg_popqa_list = sorted(union_sg_popqa)

int_sg_popqa = set()
for model, data in pqa_res.items():
    int_sg_popqa &= set(data['has_answers']['sg_dialect'])

# get a sorted list (if you want it ordered)
int_sg_popqa_list = sorted(int_sg_popqa)

print("Union of all models, sg_dialect with answers:", len(union_sg_popqa_list))

print("Intersect of all models, sg_dialect with answers:", len(int_sg_popqa_list))

Union of all models, sg_dialect with answers: 355
Intersect of all models, sg_dialect with answers: 0


In [218]:
# run track over only your union_sg_popqa_list
res_2_pop = track('POPQA', union_sg_popqa_list)

# now, for each model, count how many unique tree_ids were processed
for model, data in res_2_pop.items():
    processed = set()
    # collect from both has_answers and no_answers
    for ids in data['has_answers'].values():
        processed.update(ids)
    for ids in data['no_answers'].values():
        processed.update(ids)

    print(f"{model}: {len(processed)} IDs processed")


google/gemma-3-1b-it: 354 IDs processed
google/gemma-3-12b-it: 355 IDs processed
mistralai/Mistral-7B-Instruct-v0.2: 321 IDs processed


In [234]:
pos_union_sg_pqa = set()
for model, data in pqa_res.items():
    pos_union_sg_pqa |= set(data['has_answers']['question_position_suffix;sg_dialect'])
    pos_union_sg_pqa |= set(data['has_answers']['question_position_middle;sg_dialect'])
print(len(pos_union_sg_pqa), pos_union_sg_pqa)

94 {3585, 4612, 2057, 3082, 3600, 1553, 3094, 1571, 2595, 2089, 3629, 4145, 6706, 3636, 2614, 1080, 1081, 4669, 1605, 7242, 6224, 6243, 2661, 3174, 3177, 2667, 1644, 2169, 4220, 2693, 6278, 3208, 2187, 1679, 3731, 2198, 2204, 2717, 1694, 3241, 6321, 1716, 4282, 6334, 2753, 4802, 1737, 5321, 2763, 2252, 1745, 2259, 6871, 3289, 1246, 2785, 2274, 1250, 2788, 5345, 1768, 2280, 3305, 4331, 4864, 2817, 3329, 4867, 1796, 4358, 1799, 2828, 3854, 6415, 3859, 3861, 2839, 3864, 2329, 4384, 1825, 6435, 3879, 1845, 3893, 1850, 4415, 1858, 2883, 3910, 5446, 2379, 7049, 3494}


In [239]:
union_sg_popqa_list = set()
for model, data in pqa_res.items():
    union_sg_popqa_list |= set(data['has_answers']['sg_dialect'])
print(union_sg_popqa_list)

{6147, 2057, 3082, 7178, 4112, 4113, 10257, 6163, 1041, 3094, 23, 25, 9244, 1055, 6177, 10273, 4133, 2089, 6186, 1070, 4145, 13361, 7217, 1080, 1081, 5179, 4161, 1089, 5189, 70, 5190, 1097, 7242, 9292, 6224, 5205, 1117, 5216, 5217, 6243, 13411, 5220, 4198, 103, 3174, 3177, 106, 8300, 5236, 7285, 2169, 4220, 4221, 1153, 6278, 1158, 13446, 3208, 2187, 2198, 1174, 2204, 12445, 6304, 5285, 9384, 1193, 3241, 13488, 6321, 4282, 13501, 6334, 1220, 5321, 2252, 1228, 5325, 9423, 5328, 2259, 3289, 13532, 1246, 6367, 6368, 11486, 2274, 5345, 228, 1250, 10470, 13540, 2280, 3305, 11498, 4331, 8427, 12528, 4350, 6399, 1280, 3329, 5379, 4358, 6415, 4368, 5399, 2329, 5402, 1308, 287, 4384, 13599, 6435, 12579, 12582, 6448, 5429, 13624, 11581, 4415, 1347, 5446, 2379, 6480, 10579, 5463, 5467, 12637, 5469, 5474, 3430, 3431, 3432, 6505, 5478, 1391, 6512, 3441, 375, 8567, 6521, 4475, 12667, 5503, 7552, 2438, 3469, 402, 5524, 6549, 9624, 13724, 3494, 9641, 10667, 4524, 10668, 12715, 3500, 2481, 5554, 3507, 3

In [ ]:
intersect_sg_popqa = set(pos_union_sg_pqa) & set(union_sg_popqa_list)
print(len(intersect_sg_popqa))
print(len(pos_union_sg_pqa))
print(len(union_sg_popqa_list))

94
94
355


In [ ]:
import random

# assuming these are defined:
# union_sg_popqa_list : list of all candidate IDs
# pos_union_sg_pqa    : list or set of “positive” IDs you already have

# ensure we’re working with sets for difference
all_ids = set(union_sg_popqa_list)
pos_ids = set(pos_union_sg_pqa)

# how many more we need
n_neg = 300 - len(pos_ids)
if n_neg < 0:
    raise ValueError(f"You already have {len(pos_ids)} positives—more than 300!")

# pool of IDs to sample from
neg_pool = list(all_ids - pos_ids)
if n_neg > len(neg_pool):
    raise ValueError(f"Not enough negatives to sample: need {n_neg}, have {len(neg_pool)}")

# sample without replacement
sampled_neg = random.sample(neg_pool, n_neg)

# final combined set/list
final_ids = list(pos_ids) + sampled_neg

print(f"  positives: {len(pos_ids)}")
print(f"  sampled negatives: {len(sampled_neg)}")
print(f"  TOTAL IDs = {len(final_ids)}")  # should be 300
import random

# assuming these are defined:
# union_sg_popqa_list : list of all candidate IDs
# pos_union_sg_pqa    : list or set of “positive” IDs you already have

# ensure we’re working with sets for difference
all_ids = set(union_sg_popqa_list)
pos_ids = set(pos_union_sg_pqa)

# how many more we need
n_neg = 300 - len(pos_ids)
if n_neg < 0:
    raise ValueError(f"You already have {len(pos_ids)} positives—more than 300!")

# pool of IDs to sample from
neg_pool = list(all_ids - pos_ids)
if n_neg > len(neg_pool):
    raise ValueError(f"Not enough negatives to sample: need {n_neg}, have {len(neg_pool)}")

# sample without replacement
sampled_neg = random.sample(neg_pool, n_neg)

# final combined set/list
final_ids = list(pos_ids) + sampled_neg

print(f"  positives: {len(pos_ids)}")
print(f"  sampled negatives: {len(sampled_neg)}")
print(f"  TOTAL IDs = {len(final_ids)}")  # should be 300


  positives: 94
  sampled negatives: 206
  TOTAL IDs = 300


In [242]:
pop_final_ids = final_ids

In [220]:
intersect_sg_popqa_copy = intersect_sg_popqa
print(intersect_sg_popqa_copy)

{3585, 6147, 4612, 2057, 3082, 7178, 5647, 4112, 1553, 4113, 1041, 3600, 6163, 3094, 23, 25, 1055, 6177, 1571, 2595, 4133, 6697, 2089, 6186, 3629, 1070, 4656, 4145, 6706, 7217, 3636, 2614, 1080, 1081, 5179, 4669, 1089, 1605, 5190, 5189, 4680, 1097, 70, 7242, 6224, 5205, 601, 6747, 5724, 1117, 5216, 5217, 6243, 5220, 2661, 3174, 4198, 103, 5737, 106, 2667, 1644, 4716, 3177, 5236, 7285, 2169, 6779, 4220, 4221, 1153, 2693, 1158, 6278, 3208, 2187, 1679, 3731, 1174, 662, 2198, 2204, 2717, 1694, 671, 6304, 4770, 5285, 3241, 1193, 6321, 1716, 5815, 4282, 6334, 4799, 2753, 5826, 4802, 1220, 1737, 5321, 2763, 2252, 5836, 1228, 5325, 5328, 1745, 4813, 2259, 6868, 6871, 3289, 6873, 1246, 6367, 4832, 737, 1250, 6368, 2274, 2788, 2785, 743, 1768, 5345, 228, 4331, 2280, 3305, 4854, 4350, 6399, 4864, 2817, 770, 1280, 4867, 3329, 4358, 5379, 772, 1796, 1799, 2828, 5901, 3854, 6415, 4368, 3859, 3861, 4886, 2839, 5399, 2329, 5402, 3864, 1308, 4384, 1825, 6435, 3879, 6954, 6448, 1845, 3893, 4917, 5944, 1

In [183]:
tqa_res = track('TQA')

In [199]:
union_sg_tqa = set()
for model, data in tqa_res.items():
    union_sg_tqa |= set(data['has_answers']['sg_dialect'])

# get a sorted list (if you want it ordered)
union_sg_tqa_list = sorted(union_sg_tqa)

print("Union of all models, sg_dialect with answers:", len(union_sg_tqa_list))

# res_2_tqa is good

Union of all models, sg_dialect with answers: 500


In [ ]:
res_2_tqa = track('TQA', union_sg_tqa_list)
# now, for each model, count how many unique tree_ids were processed
for model, data in res_2_tqa.items():
    processed = set()
    # collect from both has_answers and no_answers
    for ids in data['has_answers'].values():
        processed.update(ids)
    for ids in data['no_answers'].values():
        processed.update(ids)

    print(f"{model}: {len(processed)} IDs processed")
    
    
# res_2_tqa is good

google/gemma-3-1b-it: 500 IDs processed
google/gemma-3-12b-it: 500 IDs processed
mistralai/Mistral-7B-Instruct-v0.2: 398 IDs processed


In [210]:
the_list = []
type(tqa_applied_map)
for k in tqa_applied_map:
    print(tqa_applied_map[k])
    the_list.append(list(tqa_applied_map[k]))

{45070, 36889, 38950, 4135, 30768, 36912, 32821, 4150, 2103, 30782, 24646, 53321, 75, 61520, 10326, 39007, 47205, 49255, 43112, 22639, 53359, 26740, 55420, 26750, 63617, 55426, 30858, 61583, 16533, 2203, 65693, 22690, 49323, 34992, 22710, 2232, 67770, 12489, 22734, 65745, 73939, 73940, 213, 22746, 24795, 67802, 225, 12531, 57588, 28924, 39169, 24835, 71946, 57618, 51481, 16676, 293, 6440, 57646, 2352, 55603, 69941, 20790, 22838, 26934, 8509, 72017, 14682, 61788, 61789, 74082, 39276, 18801, 55667, 63867, 67970, 2435, 24964, 43399, 4488, 37256, 14743, 67994, 70043, 55716, 12711, 51629, 18862, 41390, 49592, 55738, 43454, 18879, 57794, 22979, 12741, 51654, 61906, 47571, 37337, 41434, 18912, 51680, 485, 57832, 12777, 49644, 4589, 35320, 2555, 23038, 51718, 6663, 6665, 47626, 57867, 10768, 25104, 4638, 545, 21025, 547, 59948, 27183, 8755, 29246, 33358, 35410, 16979, 27219, 597, 64090, 62043, 21088, 27234, 37475, 21093, 66150, 43629, 43631, 43647, 8838, 41606, 10892, 53900, 66202, 8868, 31396

In [211]:
intersect_sg_tqa = set(the_list[0]) & set(union_sg_tqa_list)
len(intersect_sg_tqa)

443

In [ ]:
print(intersect_sg_tqa) #GOOD

{45070, 36889, 38950, 4135, 30768, 36912, 32821, 4150, 2103, 30782, 24646, 53321, 61520, 10326, 39007, 47205, 49255, 43112, 22639, 53359, 26740, 55420, 26750, 63617, 55426, 30858, 61583, 16533, 65693, 22690, 49323, 34992, 22710, 2232, 67770, 12489, 22734, 65745, 73939, 73940, 22746, 24795, 67802, 225, 57588, 28924, 39169, 24835, 71946, 57618, 51481, 16676, 57646, 55603, 69941, 20790, 22838, 26934, 8509, 72017, 61788, 61789, 74082, 39276, 18801, 55667, 63867, 67970, 2435, 24964, 43399, 4488, 37256, 14743, 67994, 70043, 55716, 12711, 51629, 18862, 41390, 49592, 55738, 43454, 18879, 57794, 22979, 12741, 51654, 61906, 47571, 37337, 41434, 18912, 51680, 485, 57832, 12777, 49644, 4589, 35320, 2555, 23038, 51718, 47626, 57867, 25104, 4638, 545, 21025, 547, 59948, 27183, 8755, 29246, 33358, 35410, 16979, 27219, 64090, 62043, 21088, 27234, 37475, 21093, 66150, 43629, 43631, 43647, 8838, 41606, 10892, 53900, 66202, 31396, 41638, 64177, 19126, 27324, 51912, 17103, 51928, 66277, 37607, 31467, 8955

In [214]:
pos_union_sg_tqa = set()
for model, data in tqa_res.items():
    pos_union_sg_tqa |= set(data['has_answers']['question_position_suffix;sg_dialect'])
    pos_union_sg_tqa |= set(data['has_answers']['question_position_middle;sg_dialect'])

# get a sorted list (if you want it ordered)
pos_union_sg_tqa = sorted(pos_union_sg_tqa)

print("Union of all models, sg_dialect with answers:", len(pos_union_sg_tqa))

# res_2_tqa is good

Union of all models, sg_dialect with answers: 69


In [ ]:
pos_union_sg_tqa

In [244]:
import random

# assuming these are defined:
# union_sg_popqa_list : list of all candidate IDs
# pos_union_sg_pqa    : list or set of “positive” IDs you already have

# ensure we’re working with sets for difference
all_ids = set(intersect_sg_tqa)
pos_ids = set(pos_union_sg_tqa)

# how many more we need
n_neg = 300 - len(pos_ids)
if n_neg < 0:
    raise ValueError(f"You already have {len(pos_ids)} positives—more than 300!")

# pool of IDs to sample from
neg_pool = list(all_ids - pos_ids)
if n_neg > len(neg_pool):
    raise ValueError(f"Not enough negatives to sample: need {n_neg}, have {len(neg_pool)}")

# sample without replacement
sampled_neg = random.sample(neg_pool, n_neg)

# final combined set/list
final_ids = list(pos_ids) + sampled_neg

print(f"  positives: {len(pos_ids)}")
print(f"  sampled negatives: {len(sampled_neg)}")
print(f"  TOTAL IDs = {len(final_ids)}")  # should be 300
import random

# assuming these are defined:
# union_sg_popqa_list : list of all candidate IDs
# pos_union_sg_pqa    : list or set of “positive” IDs you already have

# ensure we’re working with sets for difference
all_ids = set(intersect_sg_tqa)
pos_ids = set(pos_union_sg_tqa)

# how many more we need
n_neg = 300 - len(pos_ids)
if n_neg < 0:
    raise ValueError(f"You already have {len(pos_ids)} positives—more than 300!")

# pool of IDs to sample from
neg_pool = list(all_ids - pos_ids)
if n_neg > len(neg_pool):
    raise ValueError(f"Not enough negatives to sample: need {n_neg}, have {len(neg_pool)}")

# sample without replacement
sampled_neg = random.sample(neg_pool, n_neg)

# final combined set/list
final_ids_tqa = list(pos_ids) + sampled_neg

print(f"  positives: {len(pos_ids)}")
print(f"  sampled negatives: {len(sampled_neg)}")
print(f"  TOTAL IDs = {len(final_ids_tqa)}")  # should be 300


  positives: 69
  sampled negatives: 231
  TOTAL IDs = 300
  positives: 69
  sampled negatives: 231
  TOTAL IDs = 300


# merge trees

In [457]:
import os
import pickle
from collections import deque
from tree.tree import ReadTree
from tree.node import TerminalNode  # adjust import path as needed

required = {'question_position_suffix', 'question_position_middle', 'question_position_middle;sg_dialect', 'question_position_suffix;sg_dialect', 'sg_dialect'}

def save_tree(file_path, **kwargs):
    # Move the entire root to CPU if needed
    root = kwargs.get("root")
    if hasattr(root, "move_to_cpu"):
        root.move_to_cpu()

    node = {
        "root":             root,
        "thresholds":       kwargs.get("thresholds"),
        "prompt_list":      kwargs.get("prompt_list"),
        "time_semantic":    kwargs.get("time_semantic"),
        "time_syntactic":   kwargs.get("time_syntactic"),
        "time_check":       kwargs.get("time_check"),
        "metrics":          kwargs.get("metrics"),
        "root_prompt":      kwargs.get("root_prompt"),
        "possible_answers": kwargs.get("possible_answers"),
        "rag_entities":     kwargs.get("rag_entities"),
        "ner_entities":     kwargs.get("ner_entities"),
        "rag_closest_match": getattr(root, "rag_closest_match", None),
        "gt_passage":       kwargs.get("gt_passage"),
    }

    out_dir = os.path.dirname(file_path)
    os.makedirs(out_dir, exist_ok=True)

    with open(file_path, "wb") as fp:
        print(file_path)
        pickle.dump(node, fp)


def merge_children_trees(gen_tree_path, ref_tree_path, save_path, reset_ans=False):
    # Load both trees
    error = 0
    try:
        gen_tree = ReadTree.load_read_tree(gen_tree_path)
    except Exception as e:
        print(f"Failed to load generated tree: {e}")
        return

    try:
        ref_tree = ReadTree.load_read_tree(ref_tree_path)
    except Exception as e:
        print(f"Failed to load reference tree: {e}")
        return

    # BFS‐style merge of terminals
    queue = deque([(gen_tree.root, ref_tree.root, 0)])
    while queue:
        node_gen, node_ref, layer = queue.popleft()

        # Split children into non‐terminals vs terminals
        gen_nonterms = [c for c in node_gen.children if not isinstance(c, TerminalNode)]
        gen_terms    = [c for c in node_gen.children if isinstance(c, TerminalNode)]
        ref_terms    = [c for c in node_ref.children if isinstance(c, TerminalNode)]

        # Which ref terminals are missing?
        gen_term_names = {t.metadata.get("terminal_name") for t in gen_terms}
        ref_terms_names = {t.metadata.get("terminal_name") for t in ref_terms}
        
        for term in ref_terms:
            ref_name = term.metadata.get("terminal_name")
            # only append if it's missing *and* in your allowed set
            if ref_name not in gen_term_names and ref_name in required:
                if reset_ans:
                    term.answers = {}
                node_gen.children.append(term)
        
           
        # regen
        gen_terms = [c for c in node_gen.children if isinstance(c, TerminalNode)]
        gen_term_names = {t.metadata.get("terminal_name") for t in gen_terms}

        if not required.issubset(gen_term_names):
            missing = required - gen_term_names
            error += 1

        # Enqueue paired non‐terminals (assumes same structure)
        ref_nonterms = [c for c in node_ref.children if not isinstance(c, TerminalNode)]
        assert len(gen_nonterms) == len(ref_nonterms), (
            f"Structure mismatch at {node_gen} vs {node_ref}: "
            f"{len(gen_nonterms)} vs {len(ref_nonterms)}"
        )
        for g, r, l in zip(gen_nonterms, ref_nonterms, repeat(layer + 1)):
            queue.append((g, r, l))

    # Finally, save the merged tree using gen_tree’s metadata
    save_tree(
        save_path,
        root=gen_tree.root,
        thresholds=gen_tree.thresholds,
        prompt_list=gen_tree.prompt_list,
        time_semantic=gen_tree.time_semantic,
        time_syntactic=gen_tree.time_syntactic,
        time_check=gen_tree.time_check,
        metrics=gen_tree.metrics,
        root_prompt=gen_tree.root_prompt,
        possible_answers=gen_tree.possible_answers,
        rag_entities=gen_tree.rag_entities,
        ner_entities=gen_tree.ner_entities,
        gt_passage=gen_tree.gt_passage,
    )
    return error


In [299]:
from adapters.TerminalPerturb import TerminalPerturber
from tree.node import Node
from adapters.SemanticPerturb import PromptPackage

def generate_terminal_node(
    tree: Tree,
    parent_node: Node,
    term_perturber: TerminalPerturber,
    use_parent_passage=True,
):
    # take parent closest match
    if not use_parent_passage:
        # TODO: hehe
        raise NotImplementedError()
    new_state = {**parent_node.metadata}
    perturb_pkg = PromptPackage(text=parent_node.prompt, state=new_state)
    try:
        new_perturb_pkg: PromptPackage = term_perturber.terminal_perturb(perturb_pkg)
        # unpack results
        perturbation = new_perturb_pkg.text  # the new prompt
        perturb_state = new_perturb_pkg.state  # metadata collected by perturbers
        is_valid = perturb_state.get("is_valid", True)
    except RuntimeError as e:
        print(f"RuntimeError: {e} from prompt {perturb_pkg.text}\n {perturb_pkg.state}")
        is_valid = False
        perturb_state = perturb_pkg.state
        perturbation = perturb_pkg.text
    except AssertionError as e:
        print(f"AssertionError: {e} from prompt {perturb_pkg.text}\n {perturb_pkg.state}")
        is_valid = False
        perturb_state = perturb_pkg.state
        perturbation = perturb_pkg.text

    if is_valid:
        perturb_embedding = tree.embed_model.encode(perturbation)
        sem_sim = -100
        root_sim = -100 
        rag_closest_match = parent_node.rag_closest_match
        rag_entities = parent_node.rag_entities
        ner_entities = parent_node.ner_entities
        
        wiki_title = parent_node.wiki_title
        fk_score = textstat.flesch_kincaid_grade(perturbation)
        dc_score = textstat.dale_chall_readability_score(perturbation)
        complexity_score = (fk_score + dc_score) / 2
        term_node = TerminalNode(
                    perturbation,
                    sem_sim,
                    root_sim,
                    perturb_embedding,
                    rag_closest_match,
                    rag_entities,
                    ner_entities,
                    wiki_title=wiki_title,
                    parent=parent_node,
                    fk_score=fk_score,
                    dc_score=dc_score,
                    complexity_score=complexity_score,
                )
        term_node.metadata.update(perturb_state)
        print(term_node.metadata)
    else:
        term_node = None
    return (term_node, is_valid)

def apply_terminal_preturb(tree: ReadTree):
    # type_name = term_perturber.name
    root = tree.root
    level = 0
    queue = deque([(root, level)])

    while queue:  # BFS
        node, curr_level = queue.popleft()
        if isinstance(node, TerminalNode):
            continue  # Skip terminal nodes
        # elif isinstance(node, Node) and type_name in node.metadata.get("terminal_applied", []):
        #     continue  # Skip processed nodes
        elif isinstance(node, Node):
            # Generate a terminal node based on the current node
            #################################################################
            # apply to itself and terminal children
            terms = []
            candidates = [node]
            candidates.extend([child for child in node.children if isinstance(child, TerminalNode)])
            existing_candidate_terms = {c.metadata.get('terminal_name') for c in candidates if c.metadata.get('terminal_name') is not None}
            contains_sg = 'sg_dialect' in existing_candidate_terms
            missing_terms = ALLOWED_TERMS - existing_candidate_terms
            
            print(f"missing_terms: {missing_terms}, contains_sg: {contains_sg}")
            
            for cand_node in candidates:
                if cand_node.metadata.get('terminal_name') not in missing_terms:
                    print(f"have {cand_node.metadata.get('terminal_name')}")
                else:
                    print(f"get {cand_node.metadata.get('terminal_name')}")
                
                # term, is_valid = generate_terminal_node(tree, cand_node, term_perturber, use_parent_passage=True)
                # # You probably want to collect the valid terms
                # if is_valid:
                #     # Update node's metadata to include the type_name
                #     type_names = cand_node.metadata.get("terminal_applied", [])
                #     if not isinstance(type_names, list):
                #         type_names = [type_names]
                #     if type_name not in type_names:
                #         type_names.append(type_name)

                #     cand_node.metadata["terminal_applied"] = type_names

                #     # Set the metadata for the terminal node
                #     term.metadata = {
                #         **term.metadata,
                #         "level": curr_level
                #     }
                #     terms.append(term)
            #################################################################
            # Add children of the current node to the queue
            for child in node.children:
                queue.append((child, curr_level + 1))
            
            #################################################################
            for term in terms:
                node.add_child(term)
    tree.root = root 
    return tree

In [279]:
import os
import pickle
from collections import deque
from tree.tree import ReadTree
from tree.node import TerminalNode  # adjust import path as needed

ALLOWED_TERMS = {'question_position_middle;sg_dialect', 'question_position_suffix;sg_dialect', 'sg_dialect'}

def save_tree(file_path, **kwargs):
    # Move the entire root to CPU if needed
    root = kwargs.get("root")
    if hasattr(root, "move_to_cpu"):
        root.move_to_cpu()

    node = {
        "root":             root,
        "thresholds":       kwargs.get("thresholds"),
        "prompt_list":      kwargs.get("prompt_list"),
        "time_semantic":    kwargs.get("time_semantic"),
        "time_syntactic":   kwargs.get("time_syntactic"),
        "time_check":       kwargs.get("time_check"),
        "metrics":          kwargs.get("metrics"),
        "root_prompt":      kwargs.get("root_prompt"),
        "possible_answers": kwargs.get("possible_answers"),
        "rag_entities":     kwargs.get("rag_entities"),
        "ner_entities":     kwargs.get("ner_entities"),
        "rag_closest_match": getattr(root, "rag_closest_match", None),
        "gt_passage":       kwargs.get("gt_passage"),
    }

    out_dir = os.path.dirname(file_path)
    os.makedirs(out_dir, exist_ok=True)

    with open(file_path, "wb") as fp:
        print(file_path)
        pickle.dump(node, fp)


def get_allowed_terms_trees(gen_tree_path, save_path):
    # Load both trees
    try:
        gen_tree = ReadTree.load_read_tree(gen_tree_path)
    except Exception as e:
        print(f"Failed to load generated tree: {e}")
        return

    # BFS‐style merge of terminals
    queue = deque([gen_tree.root])
    while queue:
        node_gen = queue.popleft()

        # Split children into non‐terminals vs terminals
        gen_nonterms = [c for c in node_gen.children if not isinstance(c, TerminalNode)]
        gen_terms    = [c for c in node_gen.children if isinstance(c, TerminalNode)]

        # Which ref terminals are missing?
        gen_term_names = ALLOWED_TERMS - {t.metadata.get("terminal_name") for t in gen_terms}
        print(gen_term_names)
        # for term in ref_terms:
        #     name = term.metadata.get("terminal_name")
        #     # only append if it's missing *and* in your allowed set
        #     if name not in gen_term_names and name in ALLOWED_TERMS:
        #         node_gen.children.append(term)

        for g in gen_nonterms:
            queue.append(g)

    # Finally, save the merged tree using gen_tree’s metadata
    save_tree(
        save_path,
        root=gen_tree.root,
        thresholds=gen_tree.thresholds,
        prompt_list=gen_tree.prompt_list,
        time_semantic=gen_tree.time_semantic,
        time_syntactic=gen_tree.time_syntactic,
        time_check=gen_tree.time_check,
        metrics=gen_tree.metrics,
        root_prompt=gen_tree.root_prompt,
        possible_answers=gen_tree.possible_answers,
        rag_entities=gen_tree.rag_entities,
        ner_entities=gen_tree.ner_entities,
        gt_passage=gen_tree.gt_passage,
    )


In [ ]:
print(pop_final_ids)

[4864, 3585, 2817, 3329, 4612, 4867, 1796, 4358, 1799, 2057, 3082, 2828, 3854, 6415, 3600, 1553, 3859, 3861, 3094, 2839, 3864, 2329, 4384, 1825, 1571, 2595, 6435, 3879, 2089, 3629, 4145, 6706, 3636, 1845, 2614, 3893, 1080, 1081, 1850, 4669, 4415, 1858, 2883, 1605, 3910, 5446, 7242, 2379, 6224, 6243, 2661, 3174, 3177, 2667, 1644, 2169, 4220, 2693, 6278, 3208, 7049, 2187, 1679, 3731, 2198, 2204, 2717, 1694, 3494, 3241, 6321, 1716, 4282, 6334, 2753, 4802, 1737, 5321, 2763, 2252, 1745, 2259, 6871, 3289, 1246, 2785, 2274, 1250, 2788, 5345, 1768, 2280, 3305, 4331, 70, 10668, 4854, 13244, 10667, 13411, 3511, 2508, 5328, 1117, 5469, 6626, 11498, 5737, 3430, 7025, 14246, 13624, 6177, 5399, 10273, 3947, 2020, 25, 1568, 6629, 375, 4586, 1347, 1955, 6521, 4198, 2964, 4813, 10994, 7007, 933, 7552, 5027, 13213, 1495, 6399, 402, 601, 1966, 5724, 4680, 3507, 3965, 6747, 3500, 2965, 11177, 4716, 6116, 2503, 4221, 1153, 1488, 13599, 1041, 1174, 6147, 4993, 13005, 4062, 5836, 1228, 10821, 10149, 8146, 55

In [413]:
print(tqa_final_ids)

[39169, 54273, 67970, 28164, 15749, 8838, 21753, 37256, 13321, 30858, 41606, 10892, 56971, 44943, 61583, 8081, 14743, 22039, 17562, 21275, 66202, 43933, 40478, 65438, 65693, 27937, 22690, 55716, 69548, 55215, 9648, 40627, 63867, 22710, 2103, 26934, 17214, 9664, 64633, 34244, 53321, 28235, 8140, 73165, 17103, 27219, 59347, 13911, 60888, 67802, 10081, 16357, 43112, 57832, 9706, 53099, 39276, 49644, 3695, 18801, 26740, 36212, 48758, 34167, 56310, 18169, 57588, 8955, 11004, 29972, 33358, 47205, 48830, 12777, 30348, 52737, 24795, 37475, 51481, 48531, 42104, 64090, 28446, 59299, 13796, 51718, 59948, 27324, 73290, 31849, 34210, 26270, 34163, 14020, 55667, 3794, 61788, 19689, 3291, 73335, 56804, 5064, 15722, 29246, 48655, 52686, 18912, 21025, 51680, 23464, 9363, 24964, 22639, 58739, 40504, 36912, 8041, 18879, 3749, 62421, 52844, 69941, 73245, 51912, 59023, 52068, 62461, 69211, 41908, 67770, 38679, 66878, 50618, 33629, 31396, 5102, 36437, 21093, 33538, 37887, 50193, 34776, 27234, 48794, 61906, 

In [ ]:
# check root child
path = '/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/google-gemma-3-12b-it/complete/'
tree_id = 3910
test_tree_path = f'{path}{tree_id}_checked.pkl'
test_tree = ReadTree.load_read_tree(test_tree_path)
print([c.metadata.get("terminal_name") for c in test_tree.root.children if c.metadata.get("terminal_name") is not None])


['question_position_suffix', 'question_position_middle', 'question_position_suffix;question_position_middle', 'sg_dialect', 'question_position_suffix;sg_dialect', 'question_position_middle;sg_dialect']


In [391]:
from itertools import repeat 
from collections import deque
tree_id_2 = 4612
test_tree_path = f'{path}{tree_id_2}_checked.pkl'
test_tree = ReadTree.load_read_tree(test_tree_path)

queue = deque([(test_tree.root, 0)])    
while queue:
    node, layer = queue.popleft()
    for child in node.children:
        name = child.metadata.get("terminal_name")
        if name is not None:
            print(f'{layer}: {(name, child.answers)}')
        queue.extend(zip(node.children, repeat(layer + 1)))



0: ('question_position_suffix', {'google/gemma-3-1b-it': {'base': 'Michael Scott.', 'base_rag': 'The producer of Face to Face was **Cignal TV, Inc.**'}})
0: ('question_position_middle', {'google/gemma-3-1b-it': {'base': 'Michael Scott.', 'base_rag': 'The producer of Face to Face was **Cignal TV, Inc.**'}})
0: ('sg_dialect', {'google/gemma-3-1b-it': {'base': 'Joe Rogalski\n', 'base_rag': 'The producer of Face to Face was **Ray Davies**.'}})
1: ('question_position_suffix', {'google/gemma-3-1b-it': {'base': 'Martin Luther.', 'base_rag': 'The producer for Face to Face (2019 Sri Lankan film) is Shermal Dilshan.'}})
1: ('question_position_middle', {'google/gemma-3-1b-it': {'base': 'Martin Luther.', 'base_rag': 'The producer for Face to Face (2019 Sri Lankan film) is Shermal Dilshan.'}})
1: ('sg_dialect', {'google/gemma-3-1b-it': {'base': 'The producer of *Face/Friends* is **David Miller**.\n', 'base_rag': 'The film “Face to Face” was directed by Harsha Udakanda.'}})
1: ('question_position_su

In [392]:
from itertools import repeat 
from collections import deque
tree_id_2 = 4612
# your ref directory
ref_dir = '/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
test_tree_path = f'{ref_dir}{tree_id_2}.pkl'
test_tree = ReadTree.load_read_tree(test_tree_path)

queue = deque([(test_tree.root, 0)])    
while queue:
    node, layer = queue.popleft()
    for child in node.children:
        name = child.metadata.get("terminal_name")
        if name is not None:
            print(f'{layer}: {(name, child.answers)}')
        queue.extend(zip(node.children, repeat(layer + 1)))



0: ('question_position_suffix', {})
0: ('question_position_middle', {})
0: ('sg_dialect', {})
0: ('question_position_suffix;sg_dialect', {})
0: ('question_position_middle;sg_dialect', {})
1: ('question_position_suffix', {})
1: ('question_position_middle', {})
1: ('sg_dialect', {})
1: ('question_position_suffix;sg_dialect', {})
1: ('question_position_middle;sg_dialect', {})
1: ('question_position_suffix', {})
1: ('question_position_middle', {})
1: ('sg_dialect', {})
1: ('question_position_suffix;sg_dialect', {})
1: ('question_position_middle;sg_dialect', {})
1: ('question_position_suffix', {})
1: ('question_position_middle', {})
1: ('sg_dialect', {})
1: ('question_position_suffix;sg_dialect', {})
1: ('question_position_middle;sg_dialect', {})
1: ('question_position_suffix', {})
1: ('question_position_middle', {})
1: ('sg_dialect', {})
1: ('question_position_suffix;sg_dialect', {})
1: ('question_position_middle;sg_dialect', {})
1: ('question_position_suffix', {})
1: ('question_position_m

In [308]:
import os

# your ref directory
ref_dir = '/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'

# filter only those IDs for which the .pkl exists
filtered_ids = [
    tree_id
    for tree_id in pop_final_ids
    if os.path.isfile(os.path.join(ref_dir, f"{tree_id}.pkl"))
]

print(f"{len(filtered_ids)} out of {len(pop_final_ids)} IDs have a ref tree:")
print(filtered_ids)


300 out of 300 IDs have a ref tree:
[4864, 3585, 2817, 3329, 4612, 4867, 1796, 4358, 1799, 2057, 3082, 2828, 3854, 6415, 3600, 1553, 3859, 3861, 3094, 2839, 3864, 2329, 4384, 1825, 1571, 2595, 6435, 3879, 2089, 3629, 4145, 6706, 3636, 1845, 2614, 3893, 1080, 1081, 1850, 4669, 4415, 1858, 2883, 1605, 3910, 5446, 7242, 2379, 6224, 6243, 2661, 3174, 3177, 2667, 1644, 2169, 4220, 2693, 6278, 3208, 7049, 2187, 1679, 3731, 2198, 2204, 2717, 1694, 3494, 3241, 6321, 1716, 4282, 6334, 2753, 4802, 1737, 5321, 2763, 2252, 1745, 2259, 6871, 3289, 1246, 2785, 2274, 1250, 2788, 5345, 1768, 2280, 3305, 4331, 70, 10668, 4854, 13244, 10667, 13411, 3511, 2508, 5328, 1117, 5469, 6626, 11498, 5737, 3430, 7025, 14246, 13624, 6177, 5399, 10273, 3947, 2020, 25, 1568, 6629, 375, 4586, 1347, 1955, 6521, 4198, 2964, 4813, 10994, 7007, 933, 7552, 5027, 13213, 1495, 6399, 402, 601, 1966, 5724, 4680, 3507, 3965, 6747, 3500, 2965, 11177, 4716, 6116, 2503, 4221, 1153, 1488, 13599, 1041, 1174, 6147, 4993, 13005, 4062

In [ ]:
# merge 1 tree
ref = '/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/'
ref_tree = f'{ref}{tree_id}.pkl'
new_test_tree_path = f'{path}savemergesgdialectpso/{tree_id}_checked.pkl'
test_tree_path_tree = ReadTree.load_read_tree(test_tree_path)
new_tree = apply_terminal_preturb(test_tree_path_tree)


missing_terms: set(), contains_sg: True
have None
have question_position_suffix
have question_position_middle
have question_position_suffix;question_position_middle
have sg_dialect
have question_position_suffix;sg_dialect
have question_position_middle;sg_dialect
missing_terms: set(), contains_sg: True
have None
have question_position_suffix
have question_position_middle
have question_position_suffix;question_position_middle
have sg_dialect
have question_position_suffix;sg_dialect
have question_position_middle;sg_dialect
missing_terms: set(), contains_sg: True
have None
have question_position_suffix
have question_position_middle
have question_position_suffix;question_position_middle
have sg_dialect
have question_position_suffix;sg_dialect
have question_position_middle;sg_dialect
missing_terms: set(), contains_sg: True
have None
have question_position_suffix
have question_position_middle
have question_position_suffix;question_position_middle
have sg_dialect
have question_position_suffix;

In [377]:
# check root child
path = '/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/google-gemma-3-1b-it/complete/'
tree_id = 3910
test_tree_path = f'{path}{tree_id}_checked.pkl'
test_tree = ReadTree.load_read_tree(test_tree_path)
print([c.metadata.get("terminal_name") for c in test_tree.root.children if c.metadata.get("terminal_name") is not None])


['question_position_suffix', 'question_position_middle', 'sg_dialect']


In [484]:
for model in constants.MODELS:
    errors = 0
    error_trees = 0
    for tree_id in pop_final_ids:
        ref = '/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
        path = f'/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/{model.replace('/', '-')}/complete/'
        
        ref_tree_path = f'{ref}{tree_id}.pkl'
        gen_tree_path  =  f'{path}{tree_id}_checked.pkl'
        error = merge_children_trees(ref_tree_path, gen_tree_path, ref_tree_path, reset_ans=True)
        if error and isinstance(error, int):
            errors += error
            error_trees += 1


error_nodes = []
error_trees_list = []  
for model in constants.MODELS:
    errors = 0
    error_trees = 0
    for tree_id in pop_final_ids:
        ref = '/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
        path = f'/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/{model.replace('/', '-')}/complete/'
        ref_tree_path = f'{ref}{tree_id}.pkl'
        gen_tree_path  =  f'{path}{tree_id}_checked.pkl'
        error = merge_children_trees(gen_tree_path, ref_tree_path, gen_tree_path)
        if error and isinstance(error, int):
            errors += error
            error_trees += 1
    error_nodes.append(errors)
    error_trees_list.append(error_trees)


/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/4864.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/3585.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/2817.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/3329.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/4612.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/4867.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/1796.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/4358.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/1799.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/2057.pkl
/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/3082.pkl
/vol/bitbucket/lst20/POPQA_treen

In [506]:
from pathlib import Path

def count_pkl_files(dataset, ids):
    base_path = Path(f"/vol/bitbucket/lst20/{dataset}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/")
    
    dirs = {
        "mistral": base_path / "mistralai-Mistral-7B-Instruct-v0.2" / "complete",
        "gemma12b": base_path / "google-gemma-3-12b-it" / "complete",
        "gemma1b": base_path / "google-gemma-3-1b-it" / "complete",
        "tree (ref)": base_path / "tree",
    }

    for name, path in dirs.items():
        if not path.exists():
            print(f"❌ {name} path does not exist: {path}")
            continue

        if "tree" in name:
            count = sum((path / f"{id}.pkl").exists() for id in ids)
        else:
            count = sum((path / f"{id}_checked.pkl").exists() for id in ids)

        print(f"{name:<10}: {count}/{len(ids)} matching .pkl files")

# Example usage
ids = [1001, 1002, 1003, 1004, 1005]  # Replace with your actual list of IDs
count_pkl_files("TQA", tqa_final_ids)
count_pkl_files("POPQA", pop_final_ids)


mistral   : 247/300 matching .pkl files
gemma12b  : 300/300 matching .pkl files
gemma1b   : 300/300 matching .pkl files
tree (ref): 300/300 matching .pkl files
mistral   : 272/300 matching .pkl files
gemma12b  : 300/300 matching .pkl files
gemma1b   : 299/300 matching .pkl files
tree (ref): 300/300 matching .pkl files


In [508]:
from pathlib import Path

def count_pkl_files(dataset, ids):
    base_path = Path(f"/vol/bitbucket/lst20/{dataset}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/")
    
    dirs = {
        "mistral": base_path / "mistralai-Mistral-7B-Instruct-v0.2" / "complete",
        "gemma12b": base_path / "google-gemma-3-12b-it" / "complete",
        "gemma1b": base_path / "google-gemma-3-1b-it" / "complete",
        "tree (ref)": base_path / "tree",
    }

    for name, path in dirs.items():
        if not path.exists():
            print(f"❌ {name} path does not exist: {path}")
            continue
        count = len(list(path.glob("*.pkl")))
        print(f"{name:<10}: {count} .pkl files")

# Example usage
count_pkl_files("TQA", tqa_final_ids)
count_pkl_files("POPQA", pop_final_ids)


mistral   : 450 .pkl files
gemma12b  : 555 .pkl files
gemma1b   : 555 .pkl files
tree (ref): 555 .pkl files
mistral   : 514 .pkl files
gemma12b  : 591 .pkl files
gemma1b   : 570 .pkl files
tree (ref): 592 .pkl files


In [524]:
import zipfile
from pathlib import Path

def parcelate_ids_to_zips(dataset, ids):
    ids.sort()
    base_path = Path(f"/vol/bitbucket/lst20/{dataset}_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/")
    
    mistral_dir = base_path / "mistralai-Mistral-7B-Instruct-v0.2" / "complete"
    gemma_12b_dir = base_path / "google-gemma-3-12b-it" / "complete"
    gemma_1b_dir = base_path / "google-gemma-3-1b-it" / "complete"
    tree_dir = base_path / "tree"

    output_dir = Path(f"/homes/lst20/fyp/fyp_resources/LexEval-main/parcelate_to_process/{dataset}/")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Split ids
    mid = len(ids) // 2
    first_half = ids[:mid]
    second_half = ids[mid:]

    zip_path_1 = output_dir / "zip_1_all_model_outputs_and_trees.zip"
    zip_path_2 = output_dir / "zip_2_all_model_outputs_and_trees.zip"

    def write_zip(zip_path, id_list):
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for id in id_list:
                files = [
                    (mistral_dir / f"{id}_checked.pkl", "mistral"),
                    (gemma_12b_dir / f"{id}_checked.pkl", "gemma12b"),
                    (gemma_1b_dir / f"{id}_checked.pkl", "gemma1b"),
                    (tree_dir / f"{id}.pkl", "tree")
                ]
                for file_path, label in files:
                    if file_path.exists():
                        zipf.write(file_path, file_path.relative_to(base_path))
                    else:
                        print(f"⚠️ Missing {label} file for ID {id}: {file_path}")

    # Create zip_1 and zip_2
    write_zip(zip_path_1, first_half)
    write_zip(zip_path_2, second_half)

    print(f"✅ Created:\n- {zip_path_1}\n- {zip_path_2}")


In [525]:
parcelate_ids_to_zips('TQA', tqa_final_ids)
parcelate_ids_to_zips('POPQA', pop_final_ids)

⚠️ Missing mistral file for ID 225: /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/mistralai-Mistral-7B-Instruct-v0.2/complete/225_checked.pkl
⚠️ Missing mistral file for ID 2555: /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/mistralai-Mistral-7B-Instruct-v0.2/complete/2555_checked.pkl
⚠️ Missing mistral file for ID 3212: /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/mistralai-Mistral-7B-Instruct-v0.2/complete/3212_checked.pkl
⚠️ Missing mistral file for ID 7591: /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/mistralai-Mistral-7B-Instruct-v0.2/complete/7591_checked.pkl
⚠️ Missing mistral file for ID 9363: /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/mistralai-Mistral-7B-Instruct-v0.2/complete/9363_checked.pkl
⚠️ Missing mistral file for ID 10326: /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/mistralai-Mistral-7B-Instruct-v0.2/complet

In [481]:
from itertools import repeat 
from collections import deque
tree_id_2 = 4612
test_tree_path = f'/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/sssss/google-gemma-3-12b-it/complete/3329_checked.pkl'
test_tree = ReadTree.load_read_tree(test_tree_path)

queue = deque([(test_tree.root, 0)])    
while queue:
    node, layer = queue.popleft()
    for child in node.children:
        name = child.metadata.get("terminal_name")
        if name is not None:
            print(f'{layer}: {child.prompt} {(name, child.answers)}')
    queue.extend(zip(node.children, repeat(layer + 1)))



0: In what country is Wynau? ('question_position_suffix', {'google/gemma-3-12b-it': {'base': 'Switzerland.', 'base_rag': 'Switzerland.'}})
0: In what country is Wynau? ('question_position_middle', {'google/gemma-3-12b-it': {'base': 'Switzerland.', 'base_rag': 'Switzerland.'}})
0: What countrys are Wynau? ('sg_dialect', {})
0: Country one Wynau one? ('question_position_suffix;sg_dialect', {})
0: Country one Wynau one? ('question_position_middle;sg_dialect', {})
1: Which nation contains Wynau? The municipality is located on the western edge of the Langeten valley. ('question_position_suffix', {'google/gemma-3-12b-it': {'base': 'Switzerland.', 'base_rag': 'Switzerland. Wynau is in the canton of Bern.'}})
1: Which nation contains Wynau? The municipality is located on the western edge of the Langeten valley. ('question_position_middle', {'google/gemma-3-12b-it': {'base': 'Switzerland.', 'base_rag': 'Switzerland. Wynau is in the canton of Bern.'}})
1: Municipality is located the western edge

In [487]:
from itertools import repeat 
from collections import deque
tree_id_2 = 4612
# /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/39169.pkl 
test_tree_path = f'/vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/39169.pkl'
test_tree = ReadTree.load_read_tree(test_tree_path)

queue = deque([(test_tree.root, 0)])    
while queue:
    node, layer = queue.popleft()
    for child in node.children:
        name = child.metadata.get("terminal_name")
        if name is not None:
            print(f'{layer}: {child.prompt} {(name, child.answers)}')
    queue.extend(zip(node.children, repeat(layer + 1)))



0: Which horse won the 2013 Grand National at 66 to 1? ('question_position_suffix', {})
0: Which horse won the 2013 Grand National at 66 to 1? ('question_position_middle', {})
0: Horse one win the 2013 Grand National 66 1? ('sg_dialect', {})
0: Horse one win the 2013 Grand National 66 1? ('question_position_suffix;sg_dialect', {})
0: Horse one win those 2013 Grand National 66 1? ('question_position_middle;sg_dialect', {})
1: What horse achieved victory in the 2013 Grand National with odds of 66/1? Games are commonly known by various names including cosom hockey, ball hockey, floorball, or simply floor hockey. ('question_position_suffix', {})
1: What horse achieved victory in the 2013 Grand National with odds of 66/1? Games are commonly known by various names including cosom hockey, ball hockey, floorball, or simply floor hockey. ('question_position_middle', {})
1: Game are commonly known by various name cosom hockey, ball hockey, floorball, or simply floor hockey. horse one achieve vic

In [ ]:
tree = ReadTree.load_read_tree('/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/savemergesgdialectpso/4864.pkl')


In [447]:
print(error_nodes)
print(error_trees_list)

[1817, 1832, 1582]
[253, 254, 229]


In [425]:
sus = []
error_nodes_tqa = []
for model in constants.MODELS:
    errors = 0
    for tree_id in tqa_final_ids:
        ref = '/vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/'
        path = f'/vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/{model.replace('/', '-')}/complete/'
        ref_tree_path = f'{ref}{tree_id}.pkl'
        ref_tree = ReadTree.load_read_tree(ref_tree_path)
        print(list(c.metadata.get('terminal_name') for c in ref_tree.root.children if isinstance(c, TerminalNode)))
        gen_tree_path  =  f'{path}{tree_id}_checked.pkl'
        new_test_tree_path = f'{path}savemergesgdialectpso/{tree_id}_checked.pkl'
        error = merge_children_trees(gen_tree_path, ref_tree_path, new_test_tree_path)
        if error and isinstance(error, int):
            errors += error
    error_nodes_tqa.append(errors)


['question_position_suffix', 'question_position_middle', 'sg_dialect', 'question_position_suffix;sg_dialect', 'question_position_middle;sg_dialect']


TypeError: 'int' object is not iterable

In [415]:
print(error_nodes)

[0, 0, 0, 0, 0, 0]


# check

In [ ]:
from collections import deque

# load your tree
test_tree = ReadTree.load_read_tree(gen_tree_path)
# test_tree = ReadTree.load_read_tree(new_test_tree_path)

missings = []

root = test_tree.root

# the three substrings we care about
required = {'question_position_suffix', 'question_position_middle', 'sg_dialect'}

# pull out every terminal_name (defaulting to empty string)
child_names = [
    child.metadata.get("terminal_name", "")
    for child in root.children
]

# see which of our required bits we actually found
found = {
    req
    for req in required
    if any(req in name for name in child_names)
}

# if we didn’t find all three, found < required
if found != required:
    missing = required - found
    print(f"oh shit … missing {missing}")
    missings.append(gen_tree_path)
else:
    print('ok')


ok


In [346]:
  
# prepare a queue for BFS, starting at the root
queue = deque([root])    
# BFS loop
while queue:
    node = queue.popleft()
    for child in node.children:
        # if this child has a terminal_name, collect it
        name = child.metadata.get("terminal_name")
        if name is not None:
            print((child.metadata.get("terminal_name"), child.answers))
        # enqueue for further traversal
        queue.append(child)

('question_position_suffix', {'google/gemma-3-1b-it': {'base': 'Isaac Babel', 'base_rag': 'The composer of There Will Be Blood was Paul Thomas Anderson.'}})
('question_position_middle', {'google/gemma-3-1b-it': {'base': 'Isaac Babel', 'base_rag': 'The composer of There Will Be Blood was Paul Thomas Anderson.'}})
('sg_dialect', {'google/gemma-3-1b-it': {'base': 'Robert Mattus\n', 'base_rag': 'The composer of *There Will Be Blood* was Eric Schlosser.\n\n**Here’s a breakdown of the personnel involved:**\n\n*   **Rob Dukes:** Vocals\n*   **Gary Holt:** Guitars\n*   **Lee Altus:** Guitars\n*   **Jack Gibson:** Bass\n*   **Tom Hunting:** Drums\n\nLet’s remember that the **Food** section provides details about food classifications and nutritional components.'}})
('question_position_suffix', {'google/gemma-3-1b-it': {'base': "Okay, here's the information you requested:\n\n*   **Music for *There Will Be Blood***: The music was composed by **Harry Gregson-Cohen**.\n\n*   **American Express Headq